# 🍎 Apple Store — Learning SQL with DuckDB

Welcome! In this notebook we're going to pretend we run an Apple Store and explore our sales data using **SQL** (Structured Query Language).

SQL is how almost every company in the world asks questions of their data. By the end of this notebook you'll be able to:

- **SELECT** the data you want
- **Filter** rows with WHERE
- **Sort** results with ORDER BY
- **Summarise** data with GROUP BY
- **Join** tables together
- And a bunch more cool tricks!

Let's get started. 🚀

## Step 1 — Install & Import Libraries

We need two libraries:
- **`duckdb`** — a super fast SQL database that runs right here in Python (no server needed!)
- **`pandas`** — for generating our fake dataset and displaying tables nicely

In [1]:
%pip install duckdb pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb
import pandas as pd
import random
import itertools
from datetime import date, timedelta

random.seed(42)  # makes our random data the same every time we run
print("All good — libraries loaded!")

All good — libraries loaded!


## Step 2 — Build Our Product Catalogue

Every Apple Store has a product catalogue. We'll create one with all the real Apple product lines and their variants (storage, colour, chip tier, etc.).

In [3]:
# ── iPhones ──────────────────────────────────────────────────────────────────
iphone_models = {
    "iPhone 17":         {"base_price": 999,  "storage_opts": ["128GB", "256GB", "512GB"]},
    "iPhone 17 Pro":     {"base_price": 1199, "storage_opts": ["256GB", "512GB", "1TB"]},
    "iPhone 17 Pro Max": {"base_price": 1399, "storage_opts": ["256GB", "512GB", "1TB"]},
    "iPhone 17e":        {"base_price": 699,  "storage_opts": ["128GB", "256GB"]},
}
iphone_colors = ["Black", "Silver", "Blue", "Titanium", "Red"]
iphone_connectivity = ["Standard", "eSIM-only"]
storage_price_bump = {"128GB": 0, "256GB": 100, "512GB": 200, "1TB": 300}

iphone_rows = []
sku_counter = 1000
for model, info in iphone_models.items():
    for storage in info["storage_opts"]:
        for color in iphone_colors:
            for conn in iphone_connectivity:
                price = info["base_price"] + storage_price_bump.get(storage, 0)
                iphone_rows.append({
                    "product_id": f"SKU-{sku_counter}",
                    "category": "iPhone",
                    "model": model,
                    "storage": storage,
                    "color": color,
                    "variant_detail": conn,      # connectivity for iPhones
                    "price_usd": price,
                })
                sku_counter += 1

# ── MacBooks ──────────────────────────────────────────────────────────────────
macbook_models = {
    "MacBook Air (M5, 13-inch)":     {"base_price": 1099, "ram_opts": ["8GB", "16GB", "32GB"],        "storage_opts": ["256GB", "512GB", "1TB", "2TB"]},
    "MacBook Air (M5, 15-inch)":     {"base_price": 1299, "ram_opts": ["8GB", "16GB", "32GB"],        "storage_opts": ["256GB", "512GB", "1TB", "2TB"]},
    "MacBook Pro (M5 Pro, 14-inch)": {"base_price": 1999, "ram_opts": ["16GB", "32GB", "64GB"],       "storage_opts": ["512GB", "1TB", "2TB", "4TB"]},
    "MacBook Pro (M5 Max, 16-inch)": {"base_price": 3499, "ram_opts": ["32GB", "64GB"],               "storage_opts": ["1TB", "2TB", "4TB"]},
    "MacBook Neo":                   {"base_price": 799,  "ram_opts": ["8GB", "16GB"],                "storage_opts": ["256GB", "512GB"]},
}
ram_price_bump    = {"8GB": 0, "16GB": 200, "32GB": 400, "64GB": 800}
storage_price_bump_mac = {"256GB": 0, "512GB": 200, "1TB": 400, "2TB": 600, "4TB": 1000}

macbook_rows = []
for model, info in macbook_models.items():
    for ram in info["ram_opts"]:
        for storage in info["storage_opts"]:
            price = info["base_price"] + ram_price_bump[ram] + storage_price_bump_mac[storage]
            macbook_rows.append({
                "product_id": f"SKU-{sku_counter}",
                "category": "Mac Laptop",
                "model": model,
                "storage": storage,
                "color": "Space Grey",   # MacBooks come in one main colour for simplicity
                "variant_detail": f"{ram} RAM",
                "price_usd": price,
            })
            sku_counter += 1

# ── Mac Desktops ──────────────────────────────────────────────────────────────
desktop_configs = [
    {"model": "Mac Mini (M5)",          "category": "Mac Desktop", "chip": "M5",       "ram": "16GB", "storage": "256GB", "base_price": 599},
    {"model": "Mac Mini (M5)",          "category": "Mac Desktop", "chip": "M5",       "ram": "32GB", "storage": "512GB", "base_price": 799},
    {"model": "Mac Mini (M5)",          "category": "Mac Desktop", "chip": "M5 Pro",   "ram": "32GB", "storage": "512GB", "base_price": 1099},
    {"model": "Mac Studio (M5 Max)",    "category": "Mac Desktop", "chip": "M5 Max",   "ram": "32GB", "storage": "512GB", "base_price": 1999},
    {"model": "Mac Studio (M5 Max)",    "category": "Mac Desktop", "chip": "M5 Max",   "ram": "64GB", "storage": "1TB",   "base_price": 2599},
    {"model": "Mac Studio (M5 Ultra)",  "category": "Mac Desktop", "chip": "M5 Ultra", "ram": "64GB", "storage": "1TB",   "base_price": 3999},
    {"model": "Mac Studio (M5 Ultra)",  "category": "Mac Desktop", "chip": "M5 Ultra", "ram": "128GB","storage": "2TB",   "base_price": 5999},
    {"model": "Mac Pro",                "category": "Mac Desktop", "chip": "M5 Ultra", "ram": "192GB","storage": "4TB",   "base_price": 6999},
    {"model": "Mac Pro",                "category": "Mac Desktop", "chip": "M5 Ultra", "ram": "192GB","storage": "8TB",   "base_price": 9999},
]
desktop_rows = []
for cfg in desktop_configs:
    desktop_rows.append({
        "product_id": f"SKU-{sku_counter}",
        "category": cfg["category"],
        "model": cfg["model"],
        "storage": cfg["storage"],
        "color": "Silver",
        "variant_detail": f"{cfg['chip']} / {cfg['ram']} RAM",
        "price_usd": cfg["base_price"],
    })
    sku_counter += 1

# ── iPads ─────────────────────────────────────────────────────────────────────
ipad_models = {
    "iPad (10th Gen)":          {"base_price": 349,  "storage_opts": ["64GB", "256GB"]},
    "iPad (11th Gen)":          {"base_price": 449,  "storage_opts": ["128GB", "256GB", "512GB"]},
    "iPad Mini":                {"base_price": 499,  "storage_opts": ["128GB", "256GB", "512GB"]},
    "iPad Air (M4)":            {"base_price": 599,  "storage_opts": ["128GB", "256GB", "512GB", "1TB"]},
    "iPad Pro 11-inch (M4)":    {"base_price": 999,  "storage_opts": ["256GB", "512GB", "1TB", "2TB"]},
    "iPad Pro 13-inch (M4)":    {"base_price": 1299, "storage_opts": ["256GB", "512GB", "1TB", "2TB"]},
}
ipad_colors = ["Space Grey", "Silver", "Purple", "Blue"]
ipad_connectivity = ["WiFi", "WiFi + Cellular"]
ipad_storage_bump = {"64GB": 0, "128GB": 50, "256GB": 150, "512GB": 250, "1TB": 350, "2TB": 500}
cellular_bump = 150

ipad_rows = []
for model, info in ipad_models.items():
    for storage in info["storage_opts"]:
        for color in ipad_colors:
            for conn in ipad_connectivity:
                price = info["base_price"] + ipad_storage_bump[storage] + (cellular_bump if conn == "WiFi + Cellular" else 0)
                ipad_rows.append({
                    "product_id": f"SKU-{sku_counter}",
                    "category": "iPad",
                    "model": model,
                    "storage": storage,
                    "color": color,
                    "variant_detail": conn,
                    "price_usd": price,
                })
                sku_counter += 1

# ── Apple Watch ───────────────────────────────────────────────────────────────
watch_models = {
    "Apple Watch Series 11": {"base_price": 399,  "sizes": ["41mm", "45mm"], "materials": ["Aluminum", "Stainless Steel"]},
    "Apple Watch SE (3rd Gen)": {"base_price": 249, "sizes": ["40mm", "44mm"], "materials": ["Aluminum"]},
    "Apple Watch Ultra 3":   {"base_price": 799,  "sizes": ["49mm"],          "materials": ["Titanium"]},
}
watch_bands = ["Sport", "Milanese", "Leather", "Trail"]
band_price_bump = {"Sport": 0, "Milanese": 50, "Leather": 50, "Trail": 30}
material_price_bump = {"Aluminum": 0, "Stainless Steel": 200, "Titanium": 0}  # Ultra already priced in

watch_rows = []
for model, info in watch_models.items():
    for size in info["sizes"]:
        for material in info["materials"]:
            for band in watch_bands:
                price = info["base_price"] + material_price_bump[material] + band_price_bump[band]
                watch_rows.append({
                    "product_id": f"SKU-{sku_counter}",
                    "category": "Apple Watch",
                    "model": model,
                    "storage": "N/A",
                    "color": material,
                    "variant_detail": f"{size} / {band} Band",
                    "price_usd": price,
                })
                sku_counter += 1

# ── AirPods & Audio ───────────────────────────────────────────────────────────
audio_products = [
    {"model": "AirPods (4th Gen)",    "category": "Audio", "variant": "Standard Case",           "price_usd": 129},
    {"model": "AirPods (4th Gen)",    "category": "Audio", "variant": "Wireless Charging Case",  "price_usd": 179},
    {"model": "AirPods Pro (3rd Gen)","category": "Audio", "variant": "USB-C Case",              "price_usd": 249},
    {"model": "AirPods Pro (3rd Gen)","category": "Audio", "variant": "Wireless Charging Case",  "price_usd": 279},
    {"model": "AirPods Max 2",        "category": "Audio", "variant": "Space Grey",              "price_usd": 549},
    {"model": "AirPods Max 2",        "category": "Audio", "variant": "Silver",                  "price_usd": 549},
    {"model": "AirPods Max 2",        "category": "Audio", "variant": "Blue",                    "price_usd": 549},
    {"model": "AirPods Max 2",        "category": "Audio", "variant": "Green",                   "price_usd": 549},
]
audio_rows = []
for p in audio_products:
    audio_rows.append({
        "product_id": f"SKU-{sku_counter}",
        "category": p["category"],
        "model": p["model"],
        "storage": "N/A",
        "color": p["variant"] if "Max" in p["model"] else "White",
        "variant_detail": p["variant"],
        "price_usd": p["price_usd"],
    })
    sku_counter += 1

# ── Combine into one DataFrame ────────────────────────────────────────────────
products_df = pd.DataFrame(
    iphone_rows + macbook_rows + desktop_rows + ipad_rows + watch_rows + audio_rows
)
products_df.index = range(1, len(products_df) + 1)

print(f"Total product SKUs: {len(products_df)}")
products_df.head(10)

Total product SKUs: 361


,product_id,category,model,storage,color,variant_detail,price_usd
1,SKU-1000,iPhone,iPhone 17,128GB,Black,Standard,999
2,SKU-1001,iPhone,iPhone 17,128GB,Black,eSIM-only,999
3,SKU-1002,iPhone,iPhone 17,128GB,Silver,Standard,999
4,SKU-1003,iPhone,iPhone 17,128GB,Silver,eSIM-only,999
5,SKU-1004,iPhone,iPhone 17,128GB,Blue,Standard,999
6,SKU-1005,iPhone,iPhone 17,128GB,Blue,eSIM-only,999
7,SKU-1006,iPhone,iPhone 17,128GB,Titanium,Standard,999
8,SKU-1007,iPhone,iPhone 17,128GB,Titanium,eSIM-only,999
9,SKU-1008,iPhone,iPhone 17,128GB,Red,Standard,999
10,SKU-1009,iPhone,iPhone 17,128GB,Red,eSIM-only,999


## Step 3 — Create Fake Customers

In [4]:
first_names = [
    "Liam", "Olivia", "Noah", "Emma", "Oliver", "Ava", "James", "Sophia",
    "Elijah", "Isabella", "William", "Mia", "Henry", "Charlotte", "Lucas",
    "Amelia", "Benjamin", "Harper", "Theodore", "Evelyn", "Jack", "Luna",
    "Sebastian", "Camila", "Aiden", "Penelope", "Owen", "Riley", "Wyatt",
    "Layla", "Dylan", "Zoey", "Leo", "Nora", "Julian", "Lily", "Luca",
    "Eleanor", "Hudson", "Hannah", "Mateo", "Lillian", "Jackson", "Addison",
    "Daniel", "Aubrey", "Logan", "Ellie", "Gabriel", "Stella",
]
last_names = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
    "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez",
    "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin",
    "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark",
    "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King",
    "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green",
    "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell",
    "Carter", "Roberts",
]
cities_states = [
    ("New York", "NY"), ("Los Angeles", "CA"), ("Chicago", "IL"),
    ("Houston", "TX"), ("Phoenix", "AZ"), ("Philadelphia", "PA"),
    ("San Antonio", "TX"), ("San Diego", "CA"), ("Dallas", "TX"),
    ("San Jose", "CA"), ("Austin", "TX"), ("Jacksonville", "FL"),
    ("Fort Worth", "TX"), ("Columbus", "OH"), ("Charlotte", "NC"),
    ("Indianapolis", "IN"), ("San Francisco", "CA"), ("Seattle", "WA"),
    ("Denver", "CO"), ("Nashville", "TN"), ("Boston", "MA"), ("Portland", "OR"),
    ("Miami", "FL"), ("Atlanta", "GA"), ("Minneapolis", "MN"),
]

NUM_CUSTOMERS = 500
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    city, state = random.choice(cities_states)
    customers.append({
        "customer_id": i,
        "first_name": random.choice(first_names),
        "last_name":  random.choice(last_names),
        "age": random.randint(16, 72),
        "city": city,
        "state": state,
        "member_since": (date(2020, 1, 1) + timedelta(days=random.randint(0, 1460))).isoformat(),
    })

customers_df = pd.DataFrame(customers)
print(f"Customers created: {len(customers_df)}")
customers_df.head()

Customers created: 500


,customer_id,first_name,last_name,age,city,state,member_since
0,1,Sophia,Johnson,63,Boston,MA,2021-07-17
1,2,Lucas,Rodriguez,63,San Diego,CA,2020-07-28
2,3,Ellie,Wright,21,Portland,OR,2023-04-24
3,4,Noah,Johnson,21,Columbus,OH,2021-03-23
4,5,Leo,Hill,17,San Diego,CA,2023-02-23


## Step 4 — Generate Sales Transactions

In [5]:
stores = ["Online", "New York 5th Ave", "Los Angeles Grove", "Chicago Michigan Ave",
          "San Francisco Union Square", "Seattle University Village", "Austin Domain",
          "Miami Dadeland", "Boston Boylston", "Dallas NorthPark"]

# Weight certain categories to sell more (iPhones are the most popular!)
product_ids   = products_df["product_id"].tolist()
category_map  = products_df.set_index("product_id")["category"].to_dict()
price_map     = products_df.set_index("product_id")["price_usd"].to_dict()

category_weights = {
    "iPhone": 5, "Audio": 4, "iPad": 3,
    "Apple Watch": 3, "Mac Laptop": 2, "Mac Desktop": 1,
}
weights = [category_weights.get(category_map[pid], 1) for pid in product_ids]

NUM_SALES = 5000
start_date = date(2024, 1, 1)
end_date   = date(2025, 12, 31)
date_range = (end_date - start_date).days

sales = []
for i in range(1, NUM_SALES + 1):
    product_id = random.choices(product_ids, weights=weights, k=1)[0]
    unit_price = price_map[product_id]
    quantity   = random.choices([1, 2, 3], weights=[85, 12, 3])[0]
    # Slight holiday uplift: Dec & Nov get more sales
    sale_day   = random.randint(0, date_range)
    sale_date  = start_date + timedelta(days=sale_day)
    sales.append({
        "sale_id":     i,
        "sale_date":   sale_date.isoformat(),
        "product_id":  product_id,
        "customer_id": random.randint(1, NUM_CUSTOMERS),
        "quantity":    quantity,
        "unit_price":  unit_price,
        "total_price": round(unit_price * quantity, 2),
        "store":       random.choices(stores, weights=[40]+[7]*9)[0],  # Online is most common
        "channel":     "Online" if random.random() < 0.55 else "In-Store",
    })

sales_df = pd.DataFrame(sales)
print(f"Sales rows generated: {len(sales_df)}")
sales_df.head(10)

Sales rows generated: 5000


,sale_id,sale_date,product_id,customer_id,quantity,unit_price,total_price,store,channel
0,1,2024-05-07,SKU-1326,266,1,449,449,Online,Online
1,2,2024-12-20,SKU-1080,486,1,1699,1699,Dallas NorthPark,In-Store
2,3,2025-10-12,SKU-1299,490,1,1449,1449,Chicago Michigan Ave,Online
3,4,2025-05-29,SKU-1075,329,1,1599,1599,Online,In-Store
4,5,2025-09-11,SKU-1038,173,1,1299,1299,Miami Dadeland,In-Store
5,6,2024-02-16,SKU-1291,486,1,1499,1499,San Francisco Union Square,In-Store
6,7,2025-03-02,SKU-1218,317,1,799,799,San Francisco Union Square,Online
7,8,2024-11-05,SKU-1208,248,1,699,699,Online,In-Store
8,9,2025-12-06,SKU-1101,31,1,799,799,Online,Online
9,10,2024-05-27,SKU-1124,161,1,1699,1699,Miami Dadeland,Online


## Step 5 — Save CSVs & Load into DuckDB

We save each table as a CSV file and then load them into DuckDB. Think of DuckDB as a mini database that lives entirely on your laptop — no setup required!

In [6]:
# Save CSVs
products_df.to_csv("products.csv", index=False)
customers_df.to_csv("customers.csv", index=False)
sales_df.to_csv("sales.csv", index=False)
print("CSVs saved: products.csv, customers.csv, sales.csv")

# Connect to DuckDB (in-memory database)
con = duckdb.connect()

# Load CSVs as tables
con.execute("CREATE TABLE products  AS SELECT * FROM read_csv_auto('products.csv')")
con.execute("CREATE TABLE customers AS SELECT * FROM read_csv_auto('customers.csv')")
con.execute("CREATE TABLE sales     AS SELECT * FROM read_csv_auto('sales.csv')")

print("\nTables loaded into DuckDB:")
for table in ["products", "customers", "sales"]:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:12s} — {count:,} rows")

CSVs saved: products.csv, customers.csv, sales.csv

Tables loaded into DuckDB:
  products     — 361 rows
  customers    — 500 rows
  sales        — 5,000 rows


---

# 🎓 Let's Learn SQL!

A helper function so we can run SQL and see results as a neat table:

In [7]:
def sql(query):
    """Run a SQL query and return a pandas DataFrame."""
    return con.execute(query).df()

## Lesson 1 — SELECT & FROM: "Show me the data"

Every SQL query starts with two magic words:

```
SELECT  <columns you want>
FROM    <table name>
```

`*` means "give me ALL columns". Let's peek at our products table:

In [8]:
sql("""
SELECT *
FROM products
LIMIT 10
""")

,product_id,category,model,storage,color,variant_detail,price_usd
0,SKU-1000,iPhone,iPhone 17,128GB,Black,Standard,999
1,SKU-1001,iPhone,iPhone 17,128GB,Black,eSIM-only,999
2,SKU-1002,iPhone,iPhone 17,128GB,Silver,Standard,999
3,SKU-1003,iPhone,iPhone 17,128GB,Silver,eSIM-only,999
4,SKU-1004,iPhone,iPhone 17,128GB,Blue,Standard,999
5,SKU-1005,iPhone,iPhone 17,128GB,Blue,eSIM-only,999
6,SKU-1006,iPhone,iPhone 17,128GB,Titanium,Standard,999
7,SKU-1007,iPhone,iPhone 17,128GB,Titanium,eSIM-only,999
8,SKU-1008,iPhone,iPhone 17,128GB,Red,Standard,999
9,SKU-1009,iPhone,iPhone 17,128GB,Red,eSIM-only,999


In [ ]:
## 💡 TRY IT: change * to just pick the columns you care about
## e.g.  SELECT model, color, price_usd

## You can pick specific columns instead of *:
sql("""
SELECT [INSERT HERE]
FROM products
LIMIT 10
""")

,model,color,price_usd
0,iPhone 17,Black,999
1,iPhone 17,Black,999
2,iPhone 17,Silver,999
3,iPhone 17,Silver,999
4,iPhone 17,Blue,999
5,iPhone 17,Blue,999
6,iPhone 17,Titanium,999
7,iPhone 17,Titanium,999
8,iPhone 17,Red,999
9,iPhone 17,Red,999


## Lesson 2 — WHERE: "Only show me rows that match"

`WHERE` is like a filter. Only rows where the condition is **true** come back.

Common operators:
| Operator | Meaning |
|----------|---------|
| `=`      | equals |
| `!=`     | not equals |
| `>`  `<` | greater / less than |
| `>=` `<=`| greater/less than or equal |
| `LIKE 'pattern%'` | text pattern match (`%` = wildcard) |
| `ILIKE 'pattern%'` | text pattern match that's case insensitive (e.g. PATTERN, pATterN, pattern, etc.) |
| `IN ('a','b')` | matches any value in the list |
| `AND` / `OR` | combine multiple conditions |

In [11]:
# Show only iPhones
sql("""
SELECT model, storage, color, price_usd
FROM products
WHERE category = 'iPhone'
LIMIT 15
""")

,model,storage,color,price_usd
0,iPhone 17,128GB,Black,999
1,iPhone 17,128GB,Black,999
2,iPhone 17,128GB,Silver,999
3,iPhone 17,128GB,Silver,999
4,iPhone 17,128GB,Blue,999
5,iPhone 17,128GB,Blue,999
6,iPhone 17,128GB,Titanium,999
7,iPhone 17,128GB,Titanium,999
8,iPhone 17,128GB,Red,999
9,iPhone 17,128GB,Red,999


In [12]:
# Products over $2,000 — for the big spenders!
sql("""
SELECT model, variant_detail, price_usd
FROM products
WHERE price_usd > 2000
ORDER BY price_usd DESC
LIMIT 15
""")

,model,variant_detail,price_usd
0,Mac Pro,M5 Ultra / 192GB RAM,9999
1,Mac Pro,M5 Ultra / 192GB RAM,6999
2,Mac Studio (M5 Ultra),M5 Ultra / 128GB RAM,5999
3,"MacBook Pro (M5 Max, 16-inch)",64GB RAM,5299
4,"MacBook Pro (M5 Max, 16-inch)",32GB RAM,4899
5,"MacBook Pro (M5 Max, 16-inch)",64GB RAM,4899
6,"MacBook Pro (M5 Max, 16-inch)",64GB RAM,4699
7,"MacBook Pro (M5 Max, 16-inch)",32GB RAM,4499
8,"MacBook Pro (M5 Max, 16-inch)",32GB RAM,4299
9,Mac Studio (M5 Ultra),M5 Ultra / 64GB RAM,3999


In [13]:
# Using IN — find multiple categories at once
sql("""
SELECT model, category, price_usd
FROM products
WHERE category IN ('Audio', 'Apple Watch')
LIMIT 15
""")

,model,category,price_usd
0,Apple Watch Series 11,Apple Watch,399
1,Apple Watch Series 11,Apple Watch,449
2,Apple Watch Series 11,Apple Watch,449
3,Apple Watch Series 11,Apple Watch,429
4,Apple Watch Series 11,Apple Watch,599
5,Apple Watch Series 11,Apple Watch,649
6,Apple Watch Series 11,Apple Watch,649
7,Apple Watch Series 11,Apple Watch,629
8,Apple Watch Series 11,Apple Watch,399
9,Apple Watch Series 11,Apple Watch,449


In [14]:
# Using LIKE — find anything with "Pro" in the name
sql("""
SELECT model, category, price_usd
FROM products
WHERE model LIKE '%Pro%'
LIMIT 15
""")

,model,category,price_usd
0,iPhone 17 Pro,iPhone,1299
1,iPhone 17 Pro,iPhone,1299
2,iPhone 17 Pro,iPhone,1299
3,iPhone 17 Pro,iPhone,1299
4,iPhone 17 Pro,iPhone,1299
5,iPhone 17 Pro,iPhone,1299
6,iPhone 17 Pro,iPhone,1299
7,iPhone 17 Pro,iPhone,1299
8,iPhone 17 Pro,iPhone,1299
9,iPhone 17 Pro,iPhone,1299


## Lesson 3 — ORDER BY: "Sort the results"

`ORDER BY column ASC` → smallest first (A→Z, 0→9)  
`ORDER BY column DESC` → biggest first (Z→A, 9→0)

In [15]:
# Most expensive products first
sql("""
SELECT model, category, price_usd
FROM products
ORDER BY price_usd DESC
LIMIT 10
""")

,model,category,price_usd
0,Mac Pro,Mac Desktop,9999
1,Mac Pro,Mac Desktop,6999
2,Mac Studio (M5 Ultra),Mac Desktop,5999
3,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,5299
4,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,4899
5,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,4899
6,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,4699
7,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,4499
8,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,4299
9,Mac Studio (M5 Ultra),Mac Desktop,3999


In [16]:
# Cheapest products — great for gift shopping!
sql("""
SELECT model, category, price_usd
FROM products
ORDER BY price_usd ASC
LIMIT 10
""")

,model,category,price_usd
0,AirPods (4th Gen),Audio,129
1,AirPods (4th Gen),Audio,179
2,Apple Watch SE (3rd Gen),Apple Watch,249
3,Apple Watch SE (3rd Gen),Apple Watch,249
4,AirPods Pro (3rd Gen),Audio,249
5,Apple Watch SE (3rd Gen),Apple Watch,279
6,Apple Watch SE (3rd Gen),Apple Watch,279
7,AirPods Pro (3rd Gen),Audio,279
8,Apple Watch SE (3rd Gen),Apple Watch,299
9,Apple Watch SE (3rd Gen),Apple Watch,299


## Lesson 4 — Aggregate Functions: "Calculate totals, averages, counts"

Instead of showing individual rows, we can *summarise* data:

| Function | What it does |
|----------|-------------|
| `COUNT(*)` | Count how many rows |
| `SUM(col)` | Add up all values |
| `AVG(col)` | Calculate the average |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |

In [17]:
# How many sales did we make in total?
sql("""
SELECT
    COUNT(*)              AS total_sales,
    SUM(total_price)      AS total_revenue,
    ROUND(AVG(total_price), 2) AS avg_order_value,
    MIN(total_price)      AS cheapest_sale,
    MAX(total_price)      AS most_expensive_sale
FROM sales
""")

,total_sales,total_revenue,avg_order_value,cheapest_sale,most_expensive_sale
0,5000,7082671.0,1416.53,129,29997


## Lesson 5 — GROUP BY: "Calculate totals *per group*"

`GROUP BY` is one of the most powerful SQL tools. It splits data into groups and then applies an aggregate function to each group.

Think of it like: *"For EACH category, COUNT the sales"*

In [18]:
# Sales count and revenue by product category
sql("""
SELECT
    p.category,
    COUNT(*)                        AS number_of_sales,
    SUM(s.total_price)              AS total_revenue,
    ROUND(AVG(s.total_price), 2)    AS avg_sale_value
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
""")

,category,number_of_sales,total_revenue,avg_sale_value
0,iPhone,2223,3260361.0,1466.65
1,iPad,1921,2391915.0,1245.14
2,Mac Laptop,401,1088634.0,2714.80
3,Apple Watch,309,180127.0,582.94
4,Mac Desktop,23,106272.0,4620.52
5,Audio,123,55362.0,450.10


In [19]:
# Which store location makes the most money?
sql("""
SELECT
    store,
    COUNT(*)            AS transactions,
    SUM(total_price)    AS revenue
FROM sales
GROUP BY store
ORDER BY revenue DESC
""")

,store,transactions,revenue
0,Online,2036,2832759.0
1,Chicago Michigan Ave,354,514975.0
2,Boston Boylston,337,499466.0
3,Dallas NorthPark,357,490072.0
4,Miami Dadeland,331,476125.0
5,San Francisco Union Square,333,471183.0
6,Austin Domain,318,469246.0
7,New York 5th Ave,327,468410.0
8,Los Angeles Grove,288,432507.0
9,Seattle University Village,319,427928.0


## Lesson 6 — JOIN: "Combine data from two tables"

Our data lives in *separate* tables. A `JOIN` links them together using a shared key (like `product_id`).

```
sales  ──── product_id ────▶  products
       ──── customer_id ───▶  customers
```

`JOIN` (also written `INNER JOIN`) only returns rows that have a match in **both** tables.

In [20]:
# Join sales with product info to see what was actually sold
sql("""
SELECT
    s.sale_id,
    s.sale_date,
    p.model,
    p.category,
    s.quantity,
    s.total_price,
    s.channel
FROM sales s
JOIN products p ON s.product_id = p.product_id
LIMIT 15
""")

,sale_id,sale_date,model,category,quantity,total_price,channel
0,1,2024-05-07,Apple Watch Series 11,Apple Watch,1,449,Online
1,2,2024-12-20,iPhone 17 Pro Max,iPhone,1,1699,In-Store
2,3,2025-10-12,iPad Pro 13-inch (M4),iPad,1,1449,Online
3,4,2025-05-29,iPhone 17 Pro Max,iPhone,1,1599,In-Store
4,5,2025-09-11,iPhone 17 Pro,iPhone,1,1299,In-Store
5,6,2024-02-16,iPad Pro 11-inch (M4),iPad,1,1499,In-Store
6,7,2025-03-02,iPad Mini,iPad,1,799,Online
7,8,2024-11-05,iPad Mini,iPad,1,699,In-Store
8,9,2025-12-06,iPhone 17e,iPhone,1,799,Online
9,10,2024-05-27,"MacBook Air (M5, 15-inch)",Mac Laptop,1,1699,Online


In [21]:
# Join THREE tables: sales + products + customers
# Who bought what, and where are they from?
sql("""
SELECT
    c.first_name || ' ' || c.last_name   AS customer_name,
    c.city,
    p.model,
    p.category,
    s.total_price,
    s.sale_date
FROM sales s
JOIN products  p ON s.product_id  = p.product_id
JOIN customers c ON s.customer_id = c.customer_id
LIMIT 15
""")

,customer_name,city,model,category,total_price,sale_date
0,Hannah Hall,Austin,Apple Watch Series 11,Apple Watch,449,2024-05-07
1,Sophia Rodriguez,San Antonio,iPhone 17 Pro Max,iPhone,1699,2024-12-20
2,Aubrey Rivera,Philadelphia,iPad Pro 13-inch (M4),iPad,1449,2025-10-12
3,Wyatt Allen,Houston,iPhone 17 Pro Max,iPhone,1599,2025-05-29
4,Mateo Thomas,Miami,iPhone 17 Pro,iPhone,1299,2025-09-11
5,Sophia Rodriguez,San Antonio,iPad Pro 11-inch (M4),iPad,1499,2024-02-16
6,Layla Garcia,Houston,iPad Mini,iPad,799,2025-03-02
7,Isabella Young,Boston,iPad Mini,iPad,699,2024-11-05
8,Hudson Jones,Columbus,iPhone 17e,iPhone,799,2025-12-06
9,Layla Lee,San Antonio,"MacBook Air (M5, 15-inch)",Mac Laptop,1699,2024-05-27


## Lesson 7 — HAVING: "Filter *after* you've grouped"

`WHERE` filters rows **before** grouping.  
`HAVING` filters groups **after** `GROUP BY` has run.

Rule of thumb: if you're filtering on a `COUNT()`, `SUM()` etc. — use `HAVING`.

In [22]:
# Find our VIP customers — those who have made 5 or more purchases
sql("""
SELECT
    c.first_name || ' ' || c.last_name   AS customer_name,
    c.city,
    COUNT(*)        AS total_purchases,
    SUM(s.total_price) AS lifetime_spend
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.city
HAVING COUNT(*) >= 5
ORDER BY lifetime_spend DESC
LIMIT 15
""")

,customer_name,city,total_purchases,lifetime_spend
0,Owen Brown,New York,15,35733.0
1,Olivia Robinson,Columbus,19,35007.0
2,Ava Sanchez,San Francisco,6,34642.0
3,Lily Jackson,Chicago,12,32834.0
4,Elijah Ramirez,Charlotte,15,31057.0
5,Sophia Smith,Austin,16,30931.0
6,Amelia Nguyen,Jacksonville,17,29915.0
7,Wyatt Allen,Houston,16,29777.0
8,Sophia Robinson,Seattle,15,29512.0
9,Eleanor Hill,San Diego,23,29473.0


## Lesson 8 — Useful String & Date Functions

SQL has built-in helpers for working with text and dates.

In [23]:
# Monthly revenue trend — extract the year and month from sale_date
sql("""
SELECT
    STRFTIME(sale_date, '%Y-%m')    AS year_month,
    COUNT(*)                        AS transactions,
    SUM(total_price)                AS monthly_revenue
FROM sales
GROUP BY year_month
ORDER BY year_month
""")

,year_month,transactions,monthly_revenue
0,2024-01,212,295080.0
1,2024-02,181,261604.0
2,2024-03,217,293658.0
3,2024-04,211,300270.0
4,2024-05,219,371694.0
5,2024-06,232,312928.0
6,2024-07,226,308364.0
7,2024-08,212,292494.0
8,2024-09,191,258563.0
9,2024-10,238,314758.0


In [24]:
# Which day of the week do people buy most?
sql("""
SELECT
    DAYNAME(sale_date)  AS day_of_week,
    COUNT(*)            AS sales_count,
    SUM(total_price)    AS revenue
FROM sales
GROUP BY day_of_week
ORDER BY sales_count DESC
""")

,day_of_week,sales_count,revenue
0,Friday,737,1044069.0
1,Wednesday,735,1037203.0
2,Saturday,726,1054062.0
3,Tuesday,716,1041151.0
4,Sunday,708,987149.0
5,Thursday,705,989269.0
6,Monday,673,929768.0


## Lesson 9 — CASE WHEN: "If/else inside SQL"

`CASE WHEN` lets you create new columns based on conditions — like an if/else in Python.

In [25]:
# Label products as budget / mid-range / premium / ultra-premium
sql("""
SELECT
    model,
    category,
    price_usd,
    CASE
        WHEN price_usd <  500  THEN 'Budget'
        WHEN price_usd <  1200 THEN 'Mid-Range'
        WHEN price_usd <  3000 THEN 'Premium'
        ELSE                        'Ultra Premium'
    END AS price_tier
FROM products
ORDER BY price_usd
LIMIT 20
""")

,model,category,price_usd,price_tier
0,AirPods (4th Gen),Audio,129,Budget
1,AirPods (4th Gen),Audio,179,Budget
2,Apple Watch SE (3rd Gen),Apple Watch,249,Budget
3,Apple Watch SE (3rd Gen),Apple Watch,249,Budget
4,AirPods Pro (3rd Gen),Audio,249,Budget
5,Apple Watch SE (3rd Gen),Apple Watch,279,Budget
6,Apple Watch SE (3rd Gen),Apple Watch,279,Budget
7,AirPods Pro (3rd Gen),Audio,279,Budget
8,Apple Watch SE (3rd Gen),Apple Watch,299,Budget
9,Apple Watch SE (3rd Gen),Apple Watch,299,Budget


In [26]:
# How many sales fall into each price tier?
sql("""
SELECT
    CASE
        WHEN p.price_usd <  500  THEN 'Budget'
        WHEN p.price_usd <  1200 THEN 'Mid-Range'
        WHEN p.price_usd <  3000 THEN 'Premium'
        ELSE                          'Ultra Premium'
    END                     AS price_tier,
    COUNT(*)                AS sales_count,
    SUM(s.total_price)      AS total_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY price_tier
ORDER BY total_revenue DESC
""")

,price_tier,sales_count,total_revenue
0,Premium,2165,4012097.0
1,Mid-Range,2298,2333104.0
2,Ultra Premium,100,533078.0
3,Budget,437,204392.0


## Lesson 10 — Subqueries: "A query inside a query"

Sometimes you need to answer a question *using the result of another question*.  
A **subquery** is a SELECT statement nested inside another one.

In [27]:
# Find all products that cost MORE than the average product price
sql("""
SELECT model, category, price_usd
FROM products
WHERE price_usd > (SELECT AVG(price_usd) FROM products)
ORDER BY price_usd
LIMIT 15
""")

,model,category,price_usd
0,iPhone 17 Pro,iPhone,1299
1,iPhone 17 Pro,iPhone,1299
2,iPhone 17 Pro,iPhone,1299
3,iPhone 17 Pro,iPhone,1299
4,iPhone 17 Pro,iPhone,1299
5,"MacBook Air (M5, 13-inch)",Mac Laptop,1299
6,iPhone 17 Pro,iPhone,1299
7,iPad Pro 11-inch (M4),iPad,1299
8,iPhone 17 Pro,iPhone,1299
9,iPhone 17 Pro,iPhone,1299


In [28]:
# Top 5 best-selling individual product models (by revenue)
sql("""
SELECT model, category, total_revenue, units_sold
FROM (
    SELECT
        p.model,
        p.category,
        SUM(s.total_price)  AS total_revenue,
        SUM(s.quantity)     AS units_sold
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY p.model, p.category
) ranked
ORDER BY total_revenue DESC
LIMIT 10
""")

,model,category,total_revenue,units_sold
0,iPhone 17 Pro Max,iPhone,1054442.0,658.0
1,iPhone 17 Pro,iPhone,1013274.0,726.0
2,iPhone 17,iPhone,789083.0,717.0
3,iPad Pro 13-inch (M4),iPad,785833.0,467.0
4,iPad Pro 11-inch (M4),iPad,640436.0,464.0
5,iPhone 17e,iPhone,403562.0,538.0
6,iPad Air (M4),iPad,372809.0,441.0
7,"MacBook Pro (M5 Pro, 14-inch)",Mac Laptop,365280.0,120.0
8,"MacBook Pro (M5 Max, 16-inch)",Mac Laptop,271942.0,58.0
9,iPad (11th Gen),iPad,240592.0,358.0


---

## 🏋️ Challenge Questions

Try writing these queries yourself! The answers are hidden below each one.

### Challenge 1
> **Find all customers from California (CA) and count how many purchases each one has made. Show only those with more than 3 purchases, sorted by purchase count descending.**

*Hint: You'll need `JOIN`, `WHERE`, `GROUP BY`, `HAVING`, and `ORDER BY`*

In [ ]:
# Write your query here:
sql("""
SELECT
    -- your columns here
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
-- WHERE ...
-- GROUP BY ...
-- HAVING ...
-- ORDER BY ...
""")

<details>
<summary>👀 Click to reveal the answer</summary>

```sql
SELECT
    c.first_name || ' ' || c.last_name  AS customer_name,
    c.city,
    COUNT(*)                            AS purchase_count
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
WHERE c.state = 'CA'
GROUP BY c.customer_id, c.first_name, c.last_name, c.city
HAVING COUNT(*) > 3
ORDER BY purchase_count DESC
```
</details>

---

### Challenge 2
> **What is the most popular color for iPhones? (by number of units sold)**

*Hint: Join `sales` → `products`, filter to iPhones, group by color, sum quantity.*

In [ ]:
# Write your query here:
sql("""

""")

<details>
<summary>👀 Click to reveal the answer</summary>

```sql
SELECT
    p.color,
    SUM(s.quantity) AS units_sold
FROM sales s
JOIN products p ON s.product_id = p.product_id
WHERE p.category = 'iPhone'
GROUP BY p.color
ORDER BY units_sold DESC
```
</details>

---

### Challenge 3
> **Which month in 2024 had the highest total revenue? Show all months sorted.**

*Hint: Use `STRFTIME` to extract year-month, then `WHERE`, `GROUP BY`, `ORDER BY`.*

In [ ]:
# Write your query here:
sql("""

""")

<details>
<summary>👀 Click to reveal the answer</summary>

```sql
SELECT
    STRFTIME(sale_date, '%Y-%m') AS year_month,
    SUM(total_price)             AS monthly_revenue
FROM sales
WHERE YEAR(sale_date) = 2024
GROUP BY year_month
ORDER BY monthly_revenue DESC
```
</details>

---

### Bonus Challenge 🌟
> **Create a "sales leaderboard" showing the top 10 products by total revenue earned, including the product's category, model name, how many units were sold, and total revenue. Only include products that sold more than 20 units.**

In [ ]:
# Write your bonus query here:
sql("""

""")

<details>
<summary>👀 Click to reveal the bonus answer</summary>

```sql
SELECT
    p.category,
    p.model,
    SUM(s.quantity)     AS units_sold,
    SUM(s.total_price)  AS total_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_id, p.category, p.model
HAVING SUM(s.quantity) > 20
ORDER BY total_revenue DESC
LIMIT 10
```
</details>

---

## 🎉 You've learned the core of SQL!

Here's a quick cheat sheet of everything we covered:

| Clause | Purpose |
|--------|---------|
| `SELECT` | Choose which columns to show |
| `FROM` | Which table to read from |
| `WHERE` | Filter rows (before grouping) |
| `JOIN ... ON` | Combine two tables |
| `GROUP BY` | Group rows to aggregate |
| `HAVING` | Filter groups (after grouping) |
| `ORDER BY` | Sort the result |
| `LIMIT` | Only show the first N rows |
| `COUNT / SUM / AVG / MIN / MAX` | Aggregate functions |
| `CASE WHEN` | If/else logic |
| Subqueries | A query inside another query |

These same concepts work in **PostgreSQL**, **MySQL**, **SQLite**, **BigQuery**, **Snowflake** — basically everywhere! SQL is one of the most valuable skills you can have in tech. 🚀